In [6]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import sys
sys.path.append("../../")

import rateslib as rl

from Query.IRSwaps.IRSwapQuery import IRSwapQuery 
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapValue import IRSwapValue
from Query.IRSwaps.backends.rateslib.RLIRSwapCurve import RLIRSwapCurve
from Query.IRSwaps.backends.rateslib.rl_curve_definitions_map import RATESLIB_CURVE_DEFINITIONS

from Query.STIRFutures.STIRFutureQuery import STIRFutureQuery
from MDP.STIRFutures.STIRFutureMDP import STIRFutureMDP
from Query.STIRFutures.backends.rateslib.RLSTIRFuturePricer import RLSTIRFuturePricer

from MDP.IRSwaps.BARCHART_STIRF.risk import build_delta_risk_ladder, build_basis_risk_ladder 

In [8]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
stirf_mdp = STIRFutureMDP(source="BARCHART_STIRF-RL")

# Example: USD-Federal Funds-OIS Compound 1D Constant FOMC OCT26/DEC26 CURVE PHYS
- custy steepen 50k at 10.2bps printed 7/13/2026 14:30:30
- my mid: 7.00bps

In [9]:
ts = NYC_tz.localize(datetime.datetime(2026, 7, 13, 14, 30))

curve = "USD-OIS-Q12xM12STIRT-SERFFX-MIX23"
curve_handle = curve_mdp.get_pricer(request=dict(curve_name=curve, timestamp=ts))
curve_handle

RLIRSwapCurve(_rl_curve_id='USD-OIS', _rl_curve_handle=<rl.Curve:USD-OIS at 0x237c49330e0>, _meta_data={'timestamp': datetime.datetime(2026, 7, 13, 14, 30, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>), 'id': 'BARCHART_STIRF-RL-USD-OIS-Q12xM12STIRT-SERFFX-MIX23-2026-07-13 14:30:00-04:00', 'requested_curve_name': 'USD-OIS-Q12xM12STIRT-SERFFX-MIX23', 'curve_name': 'USD-OIS-Q12xM12STIRT-SERFFX-MIX23'})

In [10]:
risk = 50_000
tenor = "fomc_oct26/fomc_dec26"
query = IRSwapQuery(curve=curve, tenor=tenor, structure_kwargs={"bpv": risk}).resolve_query(
    ts, pricer_or_curve=curve_handle
)
pkg, rws = query.resolve_package(pricer_or_curve=curve_handle)

vmap = query.build_value_map(pricer_or_curve=curve_handle, package=pkg, risk_weights=rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV]:
    print(v.name, vmap.apply(value=v))

RATE 6.997513271962477
NPV -3.725290298461914e-09


# Risk Ladders

## Macro (First 12 consecutively quarterly IMM dated buckets SFR proxy using swaps)

In [11]:
risk_model_tenors = [
	"IMM_1xIMM_2",
    "IMM_2xIMM_3",
    "IMM_3xIMM_4",
    "IMM_4xIMM_5",
    "IMM_5xIMM_6",
    "IMM_6xIMM_7",
    "IMM_7xIMM_8",
    "IMM_8xIMM_9",
    "IMM_9xIMM_10",
    "IMM_10xIMM_11",
    "IMM_11xIMM_12",
    "IMM_12xIMM_13",
]
ois_macro_risk_curve, ois_macro_risk_ladder_solver = build_delta_risk_ladder(risk_model_tenors, curve_handle)
risk_pkg, _ = query.resolve_package(pricer_or_curve=ois_macro_risk_curve)
display(rl.Portfolio(risk_pkg).delta(solver=ois_macro_risk_ladder_solver).style.format("{:_.0f}"))

SUCCESS: `func_tol` reached after 3 iterations (levenberg_marquardt), `f_val`: 1.0041000717190983e-12, `time`: 0.0056s


## Macro (explictly using STIRFuture objects)

In [12]:
sfr_queries = [STIRFutureQuery(symbol=f"SFRCM{i}") for i in range(1, 13)]
sfr_risk_curve, sfr_risk_solver = build_delta_risk_ladder(
    sfr_queries, curve_handle, stirf_mdp_handle=stirf_mdp, timestamp=ts
)

risk_pkg, _ = query.resolve_package(pricer_or_curve=sfr_risk_curve)
display(rl.Portfolio(risk_pkg).delta(solver=sfr_risk_solver).style.format("{:_.0f}"))

SUCCESS: `func_tol` reached after 2 iterations (levenberg_marquardt), `f_val`: 1.850268700814882e-10, `time`: 0.0038s


## First 12 FF (explictly using STIRFuture objects )

In [13]:
ff_queries = [STIRFutureQuery(symbol=f"FFCM{i}") for i in range(1, 12)]
ff_risk_curve, ff_risk_solver = build_delta_risk_ladder(
    ff_queries, curve_handle, stirf_mdp_handle=stirf_mdp, timestamp=ts
)

risk_pkg, _ = query.resolve_package(pricer_or_curve=ff_risk_curve)
display(rl.Portfolio(risk_pkg).delta(solver=ff_risk_solver).style.format("{:_.0f}"))

SUCCESS: `func_tol` reached after 1 iterations (levenberg_marquardt), `f_val`: 2.5487351801995027e-11, `time`: 0.0535s


## Vanilla (short dated spot swap tenors)

In [27]:
risk_model_tenors = [
	"1d",
	"1w",
	"2w",
	"3w",
	"1m",
	"6w",
	"2m",
	"10w",
	"3m",
	"4m",
	"5m",
	"6m",
	"7m",
	"8m",
	"9m",
	"10m",
	"11m",
	"12m",
	"15m",
	"18m",
	"21m",
	"2y",
	"3y",
]
ois_macro_risk_curve, ois_macro_risk_ladder_solver = build_delta_risk_ladder(risk_model_tenors, curve_handle)
risk_pkg, _ = query.resolve_package(pricer_or_curve=ois_macro_risk_curve)
display(rl.Portfolio(risk_pkg).delta(solver=ois_macro_risk_ladder_solver).style.format("{:_.0f}"))

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 9.703298361709133e-13, `time`: 0.0023s
